[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P18/P18-1_ENISE_CORRIGE.ipynb)

# Préparer les données : ENISE

Chapitre 18 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** L'exploration est faite : une fiche liste les colonnes à écarter, les cases vides à remplir, les catégories à traduire en nombres, les échelles à unifier. Comment exécuter ces décisions dans le bon ordre, sans que le test s'en mêle ?

**Objectif terminal.** Ce chapitre montre comment, avec pandas et scikit-learn, pour l'analyse comme pour les modèles du livre Méca-IA, qui n'apprennent que sur ce que cette préparation leur donne.

**Objectif instrumental.**

- Juger un estimateur sur des données neuves : découpage, validation croisée.
- Encoder et mettre à l'échelle, avec des réglages appris sur l'apprentissage.
- Imputer sans fausser la dispersion, sélectionner et extraire des variables.
- Enchaîner préparation et modèle dans un `Pipeline`, sans fuite.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Le chapitre 17, dont les briques ouvrent le carnet de celui-ci.

## Source de données

| Fichier | Lignes | Colonnes | Unités |
|:-|-:|-:|:-|
| `MecaIA-Suivi_Production_Machines.xlsx` | 20000 | 14 | ancienneté en années, complexité et efficacité sans dimension, nombre de capteurs entier, utilisation en proportion, température en °C, vibrations en mm/s, consommation en kWh/h, productivité en pièces/h, indice de santé de 0 à 100 |
| `MecaIA-Qualite_soudure.xlsx` | 900 | 19 | expérience en années, température en °C, intensité en A, épaisseur en mm, vitesse en mm/min |
| `MecaIA-Surveillance_Vibratoire.xlsx` | 150 | 25 | vitesse en tr/min, charge en kN, pression en bar, débit en L/min, vibrations en mm/s, température en °C, émission acoustique en dB, courant en A, puissance en kW, durée de vie en h |

Les classeurs sont lus dans le dossier `donnees` du dépôt `Meca-PY_Ed2026` ; ouvert seul, par exemple dans Colab, le notebook les télécharge une fois depuis le dépôt public.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 18.1 | Stratifier, puis valider par groupes | 20 min |
| 18.2 | Le coût du codage one-hot | 15 min |
| 18.3 | La dispersion rétrécie par l'imputation | 20 min |
| 18.4 | Un filtre de variance sur le suivi de production | 20 min |
| 18.5 | Un score avec et sans fuite (défi) | 30 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### Lecture des jeux

Les jeux se cherchent d'abord dans le dépôt cloné, à deux dossiers au-dessus de ce notebook.

In [1]:
from pathlib import Path
import urllib.request

# Les jeux du chapitre : nom du classeur et adresse de la copie publique.
JEUX = {
    "MecaIA-Suivi_Production_Machines.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Suivi_Production_Machines.xlsx",
    "MecaIA-Qualite_soudure.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Qualite_soudure.xlsx",
    "MecaIA-Surveillance_Vibratoire.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Surveillance_Vibratoire.xlsx",
}

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-PY_Ed2026", "donnees")]
dossier = next((d for d in candidats if all((d / f).is_file() for f in JEUX)), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique des jeux depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    for fichier, adresse in JEUX.items():
        if not (dossier / fichier).is_file():
            try:
                urllib.request.urlretrieve(adresse, dossier / fichier)
            except OSError as erreur:
                raise FileNotFoundError(
                    "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
DONNEES = dossier
print("jeux lus dans le dossier des données :", ", ".join(JEUX))

jeux lus dans le dossier des données : MecaIA-Suivi_Production_Machines.xlsx, MecaIA-Qualite_soudure.xlsx, MecaIA-Surveillance_Vibratoire.xlsx


### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [2]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("matplotlib", version("matplotlib"))
print("numpy", version("numpy"))
print("pandas", version("pandas"))
print("scikit-learn", version("scikit-learn"))
print("scipy", version("scipy"))

Python 3.12.0
matplotlib 3.11.1
numpy 2.4.6
pandas 3.0.5
scikit-learn 1.9.0
scipy 1.18.0


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook lit les jeux de sa première cellule et reproduit le code des modules cités ; les téléverser sert à les importer dans vos propres cellules.

In [3]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = list(JEUX)                                  # les classeurs de la première cellule
MODULES_DU_LIVRE = ["meca_donnees.py"]                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Fonctions du module du livre (`meca_donnees.py`)

Le code ci-dessous reprend, tel qu'il est écrit dans le livre, les fonctions du module employées dans ce notebook ; ce code est fourni tel quel.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import (RFE, RFECV, SelectFromModel, SelectKBest, VarianceThreshold, f_classif,
                                       f_regression)
from sklearn.impute import KNNImputer, MissingIndicator, SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (GridSearchCV, KFold, ShuffleSplit, StratifiedKFold, cross_val_score,
                                     learning_curve, train_test_split)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import (Binarizer, FunctionTransformer, KBinsDiscretizer, LabelEncoder, MinMaxScaler,
                                   OneHotEncoder, OrdinalEncoder, PolynomialFeatures, PowerTransformer, RobustScaler,
                                   StandardScaler)


ORDRE_OPERATIONS = ["exclure", "imputer", "encoder", "mettre à l'échelle", "transformer", "extraire", "sélectionner"]


ECHELLES = {"standard": StandardScaler, "robuste": RobustScaler, "minmax": MinMaxScaler}


ORDRE_EMPLOI = [
    ("EDA-01", "cadrer", "17.1", ()),
    ("EDA-02", "lire_classeur", "17.1", ("EDA-01",)),
    ("EDA-03", "forme_tableau", "17.1", ("EDA-02",)),
    ("EDA-04", "bilan_manquants", "17.1", ("EDA-03",)),
    ("EDA-05", "colonnes_inutiles", "17.1", ("EDA-04",)),
    ("DEC-01", "decouper", "17.1", ("EDA-05",)),
    ("EDA-06", "tracer_distributions", "17.2", ("DEC-01",)),
    ("EDA-07", "resume_numerique", "17.2", ("EDA-06",)),
    ("EDA-09", "correlations", "17.3", ("EDA-07",)),
    ("EDA-10", "relations_paires", "17.3", ("EDA-09",)),
    ("EDA-15", "liens_entre", "17.3", ("EDA-10",)),
    ("EDA-08", "tableau_croise", "17.3", ("EDA-15",)),
    ("EDA-12", "variable_deterministe", "17.3", ("EDA-08",)),
    ("EDA-11", "aberrants_tukey", "17.4", ("EDA-12",)),
    ("EDA-13", "fiche_decisions", "17.5", ("EDA-11",)),
    ("EDA-14", "mesures_datees", "9.1", ("EDA-02",)),
    ("DEC-02", "preprocessing_Xy", "18.1", ("EDA-13", "DEC-01")),
    ("DEC-03", "decouper_stratifie", "18.1", ("DEC-02",)),
    ("DEC-04", "sensibilite_split", "18.1", ("DEC-03",)),
    ("DEC-05", "separateurs_validation", "18.1", ("DEC-04",)),
    ("DEC-06", "evaluation_lc", "18.1", ("DEC-05",)),
    ("E-1", "encoder_cible", "18.2", ("DEC-06",)),
    ("E-2", "encoder_variables", "18.2", ("E-1",)),
    ("E-3", "encoder_manuel", "18.2", ("E-2",)),
    ("N-1", "pourquoi_echelles", "18.2", ("E-3",)),
    ("N-2", "mise_echelle_minmax", "18.2", ("N-1",)),
    ("N-3", "mise_echelle_standard", "18.2", ("N-2",)),
    ("N-4", "choisir_echelle", "18.2", ("N-3", "EDA-11")),
    ("I-1", "imputer_statistique", "18.3", ("N-4",)),
    ("I-2", "imputer_voisins", "18.3", ("I-1",)),
    ("I-3", "indicateur_manquants", "18.3", ("I-2",)),
    ("S-1", "filtrer_variance", "18.4", ("I-3",)),
    ("S-2", "selection_score", "18.4", ("S-1",)),
    ("S-3", "selection_modele", "18.4", ("S-2",)),
    ("S-4", "elimination_recursive", "18.4", ("S-3",)),
    ("S-5", "controle_selection", "18.4", ("S-4",)),
    ("X-5", "preprocessing_grandeurs", "18.4", ("S-5",)),
    ("X-1", "termes_polynomiaux", "18.4", ("X-5",)),
    ("X-2", "transformer_distribution", "18.4", ("X-1",)),
    ("X-3", "decouper_en_classes", "18.4", ("X-2",)),
    ("X-4", "transformer_fonction", "18.4", ("X-3",)),
    ("A-1", "workflow_quatre_temps", "18.5", ("X-4",)),
    ("A-2", "pipeline_simple", "18.5", ("A-1",)),
    ("A-3", "preparer", "18.5", ("A-2",)),
    ("A-4", "decrire_pipeline", "18.5", ("A-3",)),
]


def dossier_jeux():
    """Dossier des classeurs : celui de la cellule de lecture du notebook."""
    return DONNEES


def lire_classeur(source, dossier=None, **options):
    """Amont : EDA-01. Lit un classeur Excel et garde l'original : renvoie (data, df), df est une copie de travail.
    >>> import io; tampon = io.BytesIO(); JOUET.to_excel(tampon, index=False)
    >>> data, df = lire_classeur(tampon); df.shape, df is not data
    ((6, 5), True)
    """
    if isinstance(source, str):
        source = Path(dossier or dossier_jeux()) / source
    data = pd.read_excel(source, **options)
    return data, data.copy()


def colonnes_inutiles(df, seuil_vide=0.9, seuil_dominante=0.999):
    """Amont : EDA-04. Colonnes à retirer : presque vides, constantes ou presque, identifiants (une valeur par ligne).
    >>> colonnes_inutiles(JOUET)
    {'vides': [], 'constantes': ['longueur_mm'], 'identifiants': []}
    """
    vides = [c for c in df if df[c].isna().mean() > seuil_vide]
    dominante = df.apply(lambda s: s.value_counts(normalize=True).max() if s.notna().any() else 1.0)
    constantes = [c for c in df if c not in vides and dominante[c] >= seuil_dominante]
    identifiants = [c for c in df if c not in vides + constantes and df[c].nunique() == len(df)
                    and not pd.api.types.is_float_dtype(df[c])]
    return {"vides": vides, "constantes": constantes, "identifiants": identifiants}


def decouper(df, test_size=0.2, random_state=42, afficher=False):
    """Amont : EDA-05. Met le test de côté avant tout croisement avec la cible ; contrôle les tailles.
    >>> apprentissage, test = decouper(JOUET, test_size=0.5); len(apprentissage), len(test)
    (3, 3)
    """
    df_train, df_test = train_test_split(df, test_size=test_size, random_state=random_state)
    assert len(df_train) + len(df_test) == len(df) and not set(df_train.index) & set(df_test.index)
    if afficher:
        print(f"{'Train set size ':.<25} {df_train.shape}\n{'Test set size ':.<25} {df_test.shape}")
    return df_train, df_test


def variable_deterministe(df, col, cible):
    """Amont : EDA-08. Part des lignes dont la cible se déduit de la variable seule (1,0 : variable à exclure).
    >>> variable_deterministe(JOUET.assign(x=[1, 1, 1, 9, 9, 9]), "x", "etat")
    1.0
    """
    d = df[[col, cible]].dropna()
    if pd.api.types.is_numeric_dtype(d[col]):
        g = d.groupby(cible)[col].agg(["min", "median", "max"]).sort_values("median")
        coupures = np.sort(((g["max"] + g["min"].shift(-1)) / 2).dropna().to_numpy())
        predite = g.index.to_numpy()[np.searchsorted(coupures, d[col].to_numpy())]
        return float((predite == d[cible].to_numpy()).mean())
    return float(d.groupby(col)[cible].agg(lambda v: v.value_counts().iloc[0]).sum() / len(d))


def aberrants_tukey(serie, k=1.5):
    """Amont : EDA-12. Bornes de Tukey (quartiles ± k fois l'écart interquartile) et masque des valeurs hors bornes.
    >>> bas, haut, hors = aberrants_tukey(JOUET["epaisseur_mm"]); (bas, haut, int(hors.sum()))
    (1.75, 3.75, 1)
    """
    q1, q3 = serie.quantile([0.25, 0.75])
    ecart = q3 - q1
    bas, haut = q1 - k * ecart, q3 + k * ecart
    return bas, haut, (serie < bas) | (serie > haut)


def preprocessing_Xy(df_in, cible=None):
    """Amont : EDA-13, DEC-01. Sépare X et y : la dernière colonne est la cible, sauf si on la nomme.
    >>> X, y = preprocessing_Xy(JOUET, "etat"); X.shape, y.name, preprocessing_Xy(JOUET)[1].name
    ((6, 4), 'etat', 'etat')
    """
    df = df_in.copy()
    cible = df.columns[-1] if cible is None else cible
    return df.drop(columns=cible), df[cible]


def decouper_stratifie(X, y, test_size=0.2, random_state=42):
    """Amont : DEC-02. Découpage qui garde les proportions de classes ; renvoie X_train, X_test, y_train, y_test.
    >>> y = pd.Series(["a"] * 8 + ["b"] * 2); r = decouper_stratifie(pd.DataFrame({"x": range(10)}), y, 0.5)
    >>> r[3].value_counts().tolist()
    [4, 1]
    """
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)


def sensibilite_split(modele, X, y, graines=range(5), test_size=0.25):
    """Amont : DEC-03. Score de validation selon la graine du découpage ; à jouer sur l'apprentissage seul.
    >>> X = pd.DataFrame({"x": range(40)}); s = sensibilite_split(KNeighborsClassifier(3), X, pd.Series([0, 1] * 20))
    >>> len(s), bool(0 <= s.min() <= s.max() <= 1)
    (5, True)
    """
    scores = []
    for graine in graines:
        Xa, Xv, ya, yv = train_test_split(X, y, test_size=test_size, random_state=graine)
        scores.append(clone(modele).fit(Xa, ya).score(Xv, yv))
    return np.array(scores)


def separateurs_validation(modele, X, y, separateurs, scoring="accuracy", groupes=None):
    """Amont : DEC-04. Score moyen et écart de la validation croisée pour chaque séparateur (KFold, StratifiedKFold...).
    >>> X = pd.DataFrame({"x": range(40)}); r = separateurs_validation(KNeighborsClassifier(3), X, pd.Series([0, 1] * 20),
    ...                                                                 {"kfold": KFold(4)}); list(r), len(r["kfold"])
    (['kfold'], 2)
    """
    resultats = {}
    for nom, cv in separateurs.items():
        s = cross_val_score(modele, X, y, cv=cv, scoring=scoring, groups=groupes)
        resultats[nom] = (s.mean(), s.std())
    return resultats


def evaluation_lc(model, X_train, y_train, scoring="r2", ax=None):
    """Amont : DEC-05. Courbe d'apprentissage : tailles, score moyen en apprentissage, score moyen en validation.
    >>> X = pd.DataFrame({"x": np.arange(60.0)}); tailles, appr, valid = evaluation_lc(LinearRegression(), X, 2 * X["x"])
    >>> len(tailles), float(valid[-1]) > 0.99
    (5, True)
    """
    tailles, appr, valid = learning_curve(model, X_train, y_train, train_sizes=np.linspace(0.2, 1.0, 5), cv=5,
                                          scoring=scoring)
    if ax is not None:
        ax.plot(tailles, appr.mean(axis=1), color="0.6", label="apprentissage")
        ax.plot(tailles, valid.mean(axis=1), color="black", label="validation")
        ax.set_xlabel("lignes d'apprentissage"); ax.legend(frameon=False)
    return tailles, appr.mean(axis=1), valid.mean(axis=1)


def encoder_cible(y_train, y_test=None):
    """Amont : DEC-06. Encode la cible en entiers (LabelEncoder appris sur l'apprentissage) ; renvoie (encodeur, ya, yt).
    >>> enc, ya, yt = encoder_cible(pd.Series(["b", "a", "b"]), pd.Series(["a"]))
    >>> enc.classes_.tolist(), ya.tolist(), yt.tolist()
    (['a', 'b'], [1, 0, 1], [0])
    """
    transformer_Encoder = LabelEncoder().fit(y_train.unique())
    y_test_code = None if y_test is None else transformer_Encoder.transform(y_test)
    return transformer_Encoder, transformer_Encoder.transform(y_train), y_test_code


def encoder_variables(X_train, X_test, colonnes, methode="onehot"):
    """Amont : E-1. Encode les variables qualitatives (matrice creuse en one-hot) ; renvoie (encodeur, A, T).
    >>> enc, A, T = encoder_variables(JOUET, JOUET, ["materiau"]); A.shape, A.nnz
    ((6, 3), 6)
    """
    if methode == "onehot":
        transformer_Encoder = OneHotEncoder(handle_unknown="ignore")
    else:
        transformer_Encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    transformer_Encoder.fit(X_train[colonnes])
    return transformer_Encoder, transformer_Encoder.transform(X_train[colonnes]), transformer_Encoder.transform(X_test[colonnes])


def encoder_manuel(X_train, X_test, colonnes):
    """Amont : E-2. Codes pandas (cat.codes) après remplissage par le mode de l'apprentissage ; -1 si modalité inconnue.
    >>> a, t = encoder_manuel(JOUET, JOUET.iloc[:1].assign(materiau="titane"), ["materiau"])
    >>> a["materiau"].tolist(), t["materiau"].tolist()
    ([0, 1, 0, 0, 1, 0], [-1])
    """
    A, T = X_train.copy(), X_test.copy()
    for col in colonnes:
        mode = X_train[col].mode()[0]
        categories = X_train[col].fillna(mode).astype("category").cat.categories
        codes = {modalite: i for i, modalite in enumerate(categories)}
        A[col] = A[col].fillna(mode).map(codes).astype(int)
        T[col] = T[col].fillna(mode).map(codes).fillna(-1).astype(int)
    return A, T


def pourquoi_echelles(x, y):
    """Amont : E-3. Pente de la droite ajustée selon les unités : la pente change, le coefficient r reste le même.
    >>> p = pourquoi_echelles(pd.Series([1.0, 2.0, 3.0, 4.0]), pd.Series([2.0, 4.0, 6.0, 8.0])); [round(v[0], 6) for v in p.values()]
    [2e-06, 2000000.0, 2.0, 1.0]
    """
    def z(s): return (s - s.mean()) / s.std(ddof=0)
    cas = {"x fois 1000, y divisé par 1000": (x * 1000, y / 1000), "x divisé par 1000, y fois 1000": (x / 1000, y * 1000),
           "unités de départ": (x, y), "centré réduit": (z(x), z(y))}
    return {nom: (float(stats.linregress(a, b).slope), float(stats.linregress(a, b).rvalue)) for nom, (a, b) in cas.items()}


def mise_echelle_minmax(X_train, X_test):
    """Amont : N-1. MinMaxScaler appris sur l'apprentissage, vérifié contre le calcul à la main ; renvoie (scaler, A, T).
    >>> a = pd.DataFrame({"x": [0.0, 5.0, 10.0]}); s, A, T = mise_echelle_minmax(a, pd.DataFrame({"x": [15.0]}))
    >>> A["x"].tolist(), T["x"].tolist()
    ([0.0, 0.5, 1.0], [1.5])
    """
    a_la_main = (X_train - X_train.min()) / (X_train.max() - X_train.min())
    transformer_Scaler = MinMaxScaler().set_output(transform="pandas").fit(X_train)
    A, T = transformer_Scaler.transform(X_train), transformer_Scaler.transform(X_test)
    assert np.allclose(A.fillna(0), a_la_main.fillna(0)), "le calcul à la main doit égaler MinMaxScaler"
    return transformer_Scaler, A, T


def mise_echelle_standard(X_train, X_test):
    """Amont : N-2. StandardScaler appris sur l'apprentissage, vérifié contre numpy ; renvoie (scaler, A, T).
    >>> a = pd.DataFrame({"x": [1.0, 2.0, 3.0]}); s, A, T = mise_echelle_standard(a, pd.DataFrame({"x": [5.0]}))
    >>> round(float(A["x"].std(ddof=0)), 6), round(float(T["x"].iloc[0]), 4)
    (1.0, 3.6742)
    """
    x = X_train.to_numpy(float)
    a_la_main = (x - np.nanmean(x, axis=0)) / np.nanstd(x, axis=0)
    transformer_Scaler = StandardScaler().set_output(transform="pandas").fit(X_train)
    A, T = transformer_Scaler.transform(X_train), transformer_Scaler.transform(X_test)
    assert np.allclose(A.to_numpy(), a_la_main, equal_nan=True), "le calcul numpy doit égaler StandardScaler"
    return transformer_Scaler, A, T


def choisir_echelle(X_train, part_aberrants=0.01):
    """Amont : N-3, EDA-11. Par colonne : robuste (médiane, quartiles) si trop d'aberrants de Tukey, sinon standard.
    >>> choix, tr = choisir_echelle(JOUET[["epaisseur_mm", "charge_kN"]], 0.1); choix
    {'epaisseur_mm': 'robuste', 'charge_kN': 'standard'}
    """
    hors = {c: float(aberrants_tukey(X_train[c])[2].mean()) for c in X_train}
    choix = {c: "robuste" if part > part_aberrants else "standard" for c, part in hors.items()}
    groupes = [(nom, ECHELLES[nom](), [c for c in choix if choix[c] == nom]) for nom in ("standard", "robuste")]
    transformer_Scaler = ColumnTransformer([g for g in groupes if g[2]], verbose_feature_names_out=False)
    return choix, transformer_Scaler.set_output(transform="pandas")


def imputer_statistique(X_train, X_test, strategie="mean", **options):
    """Amont : N-4. Remplace les manquants par une statistique de l'apprentissage ; renvoie (imputer, A, T).
    >>> A = pd.DataFrame({"x": [1.0, np.nan, 3.0]}); imp, Xa, Xt = imputer_statistique(A, A.iloc[:1].assign(x=np.nan))
    >>> Xa["x"].tolist(), Xt["x"].tolist()
    ([1.0, 2.0, 3.0], [2.0])
    """
    transformer_Imputer = SimpleImputer(missing_values=np.nan, strategy=strategie, **options)
    transformer_Imputer.set_output(transform="pandas").fit(X_train)
    return transformer_Imputer, transformer_Imputer.transform(X_train), transformer_Imputer.transform(X_test)


def imputer_voisins(X_train, X_test, k=2):
    """Amont : I-1. KNNImputer pondéré par la distance ; colonnes déjà à la même échelle de préférence.
    >>> A = pd.DataFrame({"x": [1.0, 2.0, np.nan, 4.0], "y": [1.0, 2.0, 3.0, 4.0]}); imp, Xa, Xt = imputer_voisins(A, A, 2)
    >>> round(float(Xa["x"].iloc[2]), 2)
    3.0
    """
    transformer_Imputer = KNNImputer(missing_values=np.nan, n_neighbors=k, weights="distance")
    transformer_Imputer.set_output(transform="pandas").fit(X_train)
    return transformer_Imputer, transformer_Imputer.transform(X_train), transformer_Imputer.transform(X_test)


def indicateur_manquants(X_train, X_test, features="missing-only"):
    """Amont : I-2. Colonnes 0 et 1 qui gardent la trace des manquants, colonnes choisies sur l'apprentissage.
    >>> A = pd.DataFrame({"x": [1.0, np.nan], "y": [1.0, 2.0]}); ind, Ia, It = indicateur_manquants(A, A)
    >>> Ia.columns.tolist(), Ia["manquant_x"].tolist()
    (['manquant_x'], [0, 1])
    """
    transformer_Imputer = MissingIndicator(features=features).fit(X_train)
    noms = [f"manquant_{c}" for c in X_train.columns[transformer_Imputer.features_]]
    def trace(X): return pd.DataFrame(transformer_Imputer.transform(X).astype(int), index=X.index, columns=noms)
    return transformer_Imputer, trace(X_train), trace(X_test)


def filtrer_variance(X_train, X_test, seuil=0.01):
    """Amont : I-3. Retire les colonnes de variance sous le seuil ; colonnes à la même échelle (min-max) avant.
    >>> a = pd.DataFrame({"u": [0.0, 1.0, 0.0, 1.0], "v": [0.5, 0.5, 0.5, 0.5]}); tr, A, T = filtrer_variance(a, a)
    >>> A.columns.tolist()
    ['u']
    """
    transformer_xSelecter = VarianceThreshold(threshold=seuil).set_output(transform="pandas").fit(X_train)
    return transformer_xSelecter, transformer_xSelecter.transform(X_train), transformer_xSelecter.transform(X_test)


def selection_score(X_train, y_train, X_test, k=4, score=f_regression):
    """Amont : S-1. SelectKBest : garde les k colonnes les mieux notées par un test univarié, noté sur l'apprentissage.
    >>> a = pd.DataFrame({"u": [1.0, 2.0, 3.0, 4.0], "v": [1.0, 0.0, 1.0, 0.0]}); tr, A, T = selection_score(a, 2 * a["u"], a, 1)
    >>> A.columns.tolist()
    ['u']
    """
    transformer_xSelecter = SelectKBest(score_func=score, k=k).set_output(transform="pandas")
    transformer_xSelecter.fit(X_train, y_train)
    return transformer_xSelecter, transformer_xSelecter.transform(X_train), transformer_xSelecter.transform(X_test)


def selection_modele(X_train, y_train, X_test, modele=None, max_features=4, seuil="1.25*mean"):
    """Amont : S-2. SelectFromModel : garde les colonnes dont le poids dans le modèle dépasse le seuil.
    >>> a = pd.DataFrame({"u": [1.0, 2.0, 3.0, 4.0], "v": [1.0, 0.0, 1.0, 0.0]}); tr, A, T = selection_modele(a, 2 * a["u"], a, max_features=1)
    >>> A.columns.tolist()
    ['u']
    """
    transformer_xSelecter = SelectFromModel(LinearRegression() if modele is None else modele, threshold=seuil,
                                            max_features=max_features).set_output(transform="pandas")
    transformer_xSelecter.fit(X_train, y_train)
    return transformer_xSelecter, transformer_xSelecter.transform(X_train), transformer_xSelecter.transform(X_test)


def elimination_recursive(X_train, y_train, X_test, modele=None, n=4, cv=None):
    """Amont : S-3. RFE retient n colonnes ; avec cv, RFECV choisit le nombre de colonnes par validation croisée.
    >>> a = pd.DataFrame({"u": [1.0, 2.0, 3.0, 4.0], "v": [1.0, 0.0, 1.0, 0.0]}); tr, A, T = elimination_recursive(a, 2 * a["u"], a, n=1)
    >>> A.columns.tolist()
    ['u']
    """
    modele = LinearRegression() if modele is None else modele
    transformer_xSelecter = RFE(modele, n_features_to_select=n, step=1) if cv is None \
        else RFECV(modele, step=1, min_features_to_select=n, cv=cv)
    transformer_xSelecter.set_output(transform="pandas").fit(X_train, y_train)
    return transformer_xSelecter, transformer_xSelecter.transform(X_train), transformer_xSelecter.transform(X_test)


def controle_selection(X_train, y_train, classement, modele=None, cv=5, ax=None):
    """Amont : S-4. Score de validation croisée selon le nombre de colonnes gardées, rangées du meilleur au pire.
    >>> a = pd.DataFrame({"u": np.arange(30.0), "v": np.arange(30.0) % 3}); s = controle_selection(a, 2 * a["u"], ["u", "v"])
    >>> len(s), bool(s[0] > 0.99)
    (2, True)
    """
    modele = LinearRegression() if modele is None else modele
    scores = [cross_val_score(modele, X_train[classement[:k]], y_train, cv=cv).mean() for k in range(1, len(classement) + 1)]
    if ax is not None:
        ax.plot(range(1, len(scores) + 1), scores, marker="o", color="black")
        ax.set_xlabel("nombre de variables gardées"); ax.set_ylabel("score de validation")
    return scores


def preprocessing_grandeurs(df_in, exclure=()):
    """Amont : S-5. Nettoyage du suivi de production (nom de l'auteur) : colonnes écartées, énergie par pièce.
    >>> t = pd.DataFrame({"consommation_electrique_en_kWh_par_heure": [12.0], "productivite_moyenne_en_pieces_par_heure": [60.0]})
    >>> round(float(preprocessing_grandeurs(t)["energie_par_piece_kWh"].iloc[0]), 2)
    0.2
    """
    df = df_in.drop(columns=list(exclure)).copy()
    df["energie_par_piece_kWh"] = (df["consommation_electrique_en_kWh_par_heure"]
                                   / df["productivite_moyenne_en_pieces_par_heure"])
    return df


def termes_polynomiaux(X_train, X_test, degree=2):
    """Amont : X-5. PolynomialFeatures sans biais : puissances et produits croisés, colonnes nommées.
    >>> a = pd.DataFrame({"u": [1.0, 2.0], "v": [3.0, 4.0]}); tr, A, T = termes_polynomiaux(a, a); A.columns.tolist()
    ['u', 'v', 'u^2', 'u v', 'v^2']
    """
    transformer_xEngineer = PolynomialFeatures(degree=degree, include_bias=False)
    transformer_xEngineer.set_output(transform="pandas").fit(X_train)
    return transformer_xEngineer, transformer_xEngineer.transform(X_train), transformer_xEngineer.transform(X_test)


def transformer_distribution(X_train, X_test):
    """Amont : X-1. PowerTransformer de Yeo et Johnson : rend la distribution plus symétrique, centrée réduite.
    >>> a = pd.DataFrame({"x": [1.0, 1.1, 1.2, 1.3, 9.0]}); tr, A, T = transformer_distribution(a, a)
    >>> bool(abs(A["x"].skew()) < abs(a["x"].skew()))
    True
    """
    transformer_xEngineer = PowerTransformer(method="yeo-johnson", standardize=True)
    transformer_xEngineer.set_output(transform="pandas").fit(X_train)
    return transformer_xEngineer, transformer_xEngineer.transform(X_train), transformer_xEngineer.transform(X_test)


def decouper_en_classes(X_train, X_test, n_bins=4, strategie="kmeans", seuil=None):
    """Amont : X-2. Classes ordonnées (KBinsDiscretizer) ou deux classes autour d'un seuil (Binarizer).
    >>> a = pd.DataFrame({"x": [1.0, 2.0, 8.0, 9.0]}); tr, A, T = decouper_en_classes(a, a, seuil=5.0); A["x"].tolist()
    [0.0, 0.0, 1.0, 1.0]
    """
    if seuil is None:
        transformer_xEngineer = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy=strategie)
    else:
        transformer_xEngineer = Binarizer(threshold=seuil)
    transformer_xEngineer.set_output(transform="pandas").fit(X_train)
    return transformer_xEngineer, transformer_xEngineer.transform(X_train), transformer_xEngineer.transform(X_test)


def transformer_fonction(X_train, X_test, f, f_inverse=None):
    """Amont : X-3. Fonction choisie par l'ingénieur (FunctionTransformer), avec sa réciproque pour revenir aux unités.
    >>> a = pd.DataFrame({"x": [0.0, 4.0, 16.0]}); tr, A, T = transformer_fonction(a, a, np.sqrt, np.square)
    >>> A["x"].tolist(), tr.inverse_transform(A)["x"].tolist()
    ([0.0, 2.0, 4.0], [0.0, 4.0, 16.0])
    """
    transformer_xEngineer = FunctionTransformer(f, inverse_func=f_inverse, check_inverse=False, feature_names_out="one-to-one")
    transformer_xEngineer.set_output(transform="pandas").fit(X_train)
    return transformer_xEngineer, transformer_xEngineer.transform(X_train), transformer_xEngineer.transform(X_test)


def workflow_quatre_temps():
    """Amont : X-4. Les quatre temps de l'auteur : référence, standardisation sans pipeline, pipeline, amélioration.
    >>> [temps for temps, geste in workflow_quatre_temps()]
    ['1 Référence', '2 Standardisation sans pipeline', '3 Pipeline', "4 Boucle d'amélioration"]
    """
    return [("1 Référence", "colonnes complètes, découpage, modèle simple : le score à battre"),
            ("2 Standardisation sans pipeline", "échelle apprise sur tout l'apprentissage avant la validation croisée : les plis la partagent"),
            ("3 Pipeline", "l'échelle est apprise dans chaque pli : score sans fuite"),
            ("4 Boucle d'amélioration", "branches numérique et catégorielle, grille d'hyperparamètres, retour à l'EDA")]


def pipeline_simple(modele, X_train, y_train, grille=None, cv=5):
    """Amont : A-1. Pipeline (mise à l'échelle puis modèle) et, avec une grille, recherche d'hyperparamètres.
    >>> X = pd.DataFrame({"x": range(40)}); p = pipeline_simple(KNeighborsClassifier(3), X, pd.Series([0, 1] * 20))
    >>> list(p.named_steps)
    ['scaler', 'classifier']
    """
    pipe = Pipeline([("scaler", StandardScaler()), ("classifier", modele)])
    if grille is None:
        return pipe.fit(X_train, y_train)
    return GridSearchCV(pipe, grille, cv=cv).fit(X_train, y_train)


def preparer(num_cols, cat_cols, k="all", xeng="passthrough", score=f_regression):
    """Amont : A-2. Chaîne complète de préparation : branche numérique, branche catégorielle, k meilleures colonnes.
    >>> X, y = JOUET[["epaisseur_mm", "materiau"]], JOUET["charge_kN"]
    >>> preparer(["epaisseur_mm"], ["materiau"]).fit(X, y).transform(X).shape
    (6, 3)
    """
    num_cols_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                                    ("xeng", xeng)])
    cat_cols_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder(handle_unknown="ignore"))])
    preprocessor = ColumnTransformer(transformers=[("num_cols_preproc", num_cols_pipe, num_cols),
                                                   ("cat_cols_preproc", cat_cols_pipe, cat_cols)])
    k_best = SelectKBest(score, k=k)
    return Pipeline(steps=[("preprocessor", preprocessor), ("k_best", k_best)])


def decrire_pipeline(preparation):
    """Amont : A-3. Flowchart en texte : une ligne par branche, les étapes dans l'ordre, puis la sélection.
    >>> p = preparer(["epaisseur_mm"], ["materiau"])
    >>> decrire_pipeline(p)[0]
    'num_cols_preproc : SimpleImputer puis StandardScaler sur 1 colonnes'
    """
    branches = preparation.named_steps["preprocessor"].transformers
    lignes = [f"{nom} : {' puis '.join(type(e).__name__ for _, e in branche.steps if e != 'passthrough')} "
              f"sur {len(cols)} colonnes" for nom, branche, cols in branches]
    return lignes + ["sélection des k meilleures colonnes (SelectKBest)"]

### Préparation

In [5]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import (GridSearchCV, GroupKFold, KFold, ShuffleSplit, StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
plt.close("all")
CIBLE = "etat_sante_machine"
# Verdict du chapitre 17, rejoué ici : deux colonnes presque constantes, une variable qui fixe la cible.
data, df = lire_classeur("MecaIA-Suivi_Production_Machines.xlsx")
df_train, df_test = decouper(df)                      # même graine qu'au chapitre 17 : mêmes 16 000 et 4 000 lignes

In [6]:
EXCLURE = colonnes_inutiles(df)["constantes"] + ["indice_sante_machine_0_100"]
assert variable_deterministe(df_train, "indice_sante_machine_0_100", CIBLE) == 1.0
assert sorted(EXCLURE) == ["indice_sante_machine_0_100", "mode_de_fonctionnement",
                           "type_de_machine"]

## Exercice 18.1 : Stratifier, puis valider par groupes (20 min)

Découpez l'apprentissage du suivi de production en lots de $75$ et $25\,\%$ au hasard (graine $42$), puis de façon stratifiée : mesurez l'écart maximal de part d'un état entre les deux lots. Évaluez ensuite, avec trois voisins et les colonnes complètes, `KFold`, `StratifiedKFold` (mélangé, graine $0$) à cinq plis et `GroupKFold` à trois plis, les groupes étant les classes de productivité.

**Résultat attendu.** Écart de 1,72 point au hasard contre 0,02 point stratifié ; exactitude de 57,3 %, 57,2 % et 60,2 % par groupes, avec un écart entre plis de 2,8 points contre 1,0 pour `KFold`.

**Seconde voie.** Avec `GroupKFold`, aucun groupe n'est partagé entre apprentissage et validation : vérifiez-le sur les index.

**Indication.** Le découpage stratifié vient de `decouper_stratifie` avec `test_size=0.25`, le hasard de `train_test_split` sans `stratify` ; comparez les parts de chaque état dans les deux lots. `separateurs_validation` reçoit un dictionnaire de séparateurs ; pour `GroupKFold`, passez la colonne de classe de productivité en `groupes`. L'écart entre plis augmente quand un groupe, plus difficile que les autres, tombe seul dans un pli.

In [7]:
# Solution :
X_train, y_train = preprocessing_Xy(df_train.drop(columns=EXCLURE), CIBLE)
X_test, y_test = preprocessing_Xy(df_test.drop(columns=EXCLURE), CIBLE)
SP_NUM = X_train.select_dtypes("number").columns.tolist()
complets = [c for c in SP_NUM if not X_train[c].isna().any()]
print("X_train :", X_train.shape, " X_test :", X_test.shape, " colonnes sans manquant :", len(complets))
# Un découpage de plus, dans l'apprentissage seul : plein hasard contre stratifié (part de chaque état).
X_a, X_v, y_a, y_v = decouper_stratifie(X_train, y_train, test_size=0.25)
X_p, X_pv, y_p, y_pv = train_test_split(X_train, y_train, test_size=0.25, random_state=42)
ecart_strat = (y_a.value_counts(normalize=True) - y_v.value_counts(normalize=True)).abs().max()
ecart_hasard = (y_p.value_counts(normalize=True) - y_pv.value_counts(normalize=True)).abs().max()
print("écart maximal de part entre les deux lots : %.4f au hasard, %.4f stratifié" % (ecart_hasard, ecart_strat))
modele = KNeighborsClassifier(n_neighbors=3)
scores_graines = sensibilite_split(modele, X_train[complets], y_train)
print("exactitude de validation selon la graine :", scores_graines.round(4), " écart %.4f" % np.ptp(scores_graines))
separateurs = {"KFold": KFold(5), "KFold mélangé": KFold(5, shuffle=True, random_state=0),
               "StratifiedKFold": StratifiedKFold(5, shuffle=True, random_state=0),
               "ShuffleSplit": ShuffleSplit(5, test_size=0.2, random_state=0)}
resultats = separateurs_validation(modele, X_train[complets], y_train, separateurs)
resultats["GroupKFold"] = separateurs_validation(modele, X_train[complets], y_train, {"g": GroupKFold(3)},
                                                 groupes=X_train["classe_niveau_productivite"])["g"]

for nom, (moyenne, ecart) in resultats.items():
    print("  %-16s exactitude %.4f   écart entre plis %.4f" % (nom, moyenne, ecart))

fig, ax = plt.subplots(figsize=(3.4, 2.4))
tailles, apprentissage, validation = evaluation_lc(modele, X_train[complets], y_train, scoring="accuracy", ax=ax)
print("courbe d'apprentissage : %d lignes, exactitude %.3f en apprentissage, %.3f en validation"
      % (tailles[-1], apprentissage[-1], validation[-1]))
# Fin de la solution
assert X_train.shape == (16000, 10) and X_test.shape == (4000, 10) and len(complets) == 6
assert ecart_strat < 0.0005 < 0.005 < ecart_hasard, \
    "la stratification garde les parts à 0,05 % près"
assert 0.01 < np.ptp(scores_graines) < 0.02 and abs(scores_graines.mean() - 0.57225) < 0.0005
assert max(e for n, (m, e) in resultats.items() if n != "GroupKFold") < 0.01
assert abs(resultats["GroupKFold"][1] - 0.0284) < 0.00005, "plis par groupes : dispersion accrue"
assert apprentissage[-1] - validation[-1] > 0.14 and validation[-1] > validation[0]
# Solution :
plt.close(fig)
# Fin de la solution
print("Exercice 18.1 : contrôles passés")

X_train : (16000, 10)  X_test : (4000, 10)  colonnes sans manquant : 6
écart maximal de part entre les deux lots : 0.0172 au hasard, 0.0002 stratifié
exactitude de validation selon la graine : [0.5668 0.5825 0.5692 0.573  0.5698]  écart 0.0158
  KFold            exactitude 0.5726   écart entre plis 0.0099
  KFold mélangé    exactitude 0.5717   écart entre plis 0.0099
  StratifiedKFold  exactitude 0.5716   écart entre plis 0.0043
  ShuffleSplit     exactitude 0.5691   écart entre plis 0.0067
  GroupKFold       exactitude 0.6024   écart entre plis 0.0284
courbe d'apprentissage : 12800 lignes, exactitude 0.739 en apprentissage, 0.573 en validation
Exercice 18.1 : contrôles passés


## Exercice 18.2 : Le coût du codage one-hot (15 min)

Dans l'apprentissage du contrôle de soudures, sans les colonnes écartées, encodez toutes les colonnes de texte avec `encoder_variables`, en one-hot puis en ordinal : comparez les nombres de colonnes et les valeurs non nulles de la matrice creuse.

**Résultat attendu.** 7 colonnes de texte ; 31 colonnes en one-hot contre 7 en ordinal ; 5 040 valeurs non nulles.

**Seconde voie.** Le nombre de colonnes du one-hot est la somme des modalités, case vide comprise : `Xq[textes].nunique(dropna=False).sum()`.

**Indication.** Les colonnes de texte sont celles qui ne sont ni numériques ni booléennes : `Xq.select_dtypes(exclude=["number", "bool"]).columns`. Le one-hot crée une colonne par modalité, case vide comprise, car `OneHotEncoder` traite le manquant comme une modalité ; l'ordinal en garde une par colonne. La matrice creuse porte une valeur non nulle par ligne et par colonne de texte.

In [8]:
# Solution :
data_q, qs = lire_classeur("MecaIA-Qualite_soudure.xlsx")
CIBLE_Q = "qualite_soudure"
inutiles_q = colonnes_inutiles(qs)
qs_train, qs_test = decouper(qs)
# Fin de la solution
assert variable_deterministe(qs_train, "controle_visuel_resultat", CIBLE_Q) == 1.0, \
    "contrôle posé après la soudure"
# Solution :
EXCLURE_Q = inutiles_q["identifiants"] + inutiles_q["vides"] + ["controle_visuel_resultat"]
Xq, yq = preprocessing_Xy(qs_train.drop(columns=EXCLURE_Q), CIBLE_Q)
Xq_test, yq_test = preprocessing_Xy(qs_test.drop(columns=EXCLURE_Q), CIBLE_Q)
codeur_cible, yq_code, yq_test_code = encoder_cible(yq, yq_test)
print("cible encodée :", list(codeur_cible.classes_), " effectifs :", np.bincount(yq_code).tolist(),
      " test :", np.bincount(yq_test_code).tolist())
textes = Xq.select_dtypes(exclude=["number", "bool"]).columns.tolist()
modalites = int(Xq[textes].nunique(dropna=False).sum())
enc_un, onehot_train, onehot_test = encoder_variables(Xq, Xq_test, textes)
enc_ord, ord_train, ord_test = encoder_variables(Xq, Xq_test, textes, methode="ordinal")
print("colonnes de texte : %d ; one-hot : %d colonnes (matrice creuse, %d valeurs non nulles) ; ordinal : %d colonnes"
      % (len(textes), onehot_train.shape[1], onehot_train.nnz, ord_train.shape[1]))
manuel_train, manuel_test = encoder_manuel(Xq, Xq_test, ["type_assemblage", "site_fabrication"])
print("encodage manuel : mode de type_assemblage = %s, de site_fabrication = %s"
      % (Xq["type_assemblage"].mode()[0], Xq["site_fabrication"].mode()[0]))
pentes = pourquoi_echelles(X_train["indice_efficacite_energetique"], X_train["productivite_moyenne_en_pieces_par_heure"])

for nom, (pente, r) in pentes.items():
    print("  %-34s pente %.3e   r %.4f" % (nom, pente, r))

mm, mm_train, mm_test = mise_echelle_minmax(X_train[SP_NUM], X_test[SP_NUM])
st, st_train, st_test = mise_echelle_standard(X_train[SP_NUM], X_test[SP_NUM])
print("min-max : étendue apprise de nombre_capteurs_installes [%.0f ; %.0f], maximum du test mis à l'échelle %.3f"
      % (mm.data_min_[2], mm.data_max_[2], mm_test["nombre_capteurs_installes"].max()))
print("standard : moyenne apprise de l'ancienneté %.2f an, écart type %.2f an" % (st.mean_[0], st.scale_[0]))
choix, transformer_Scaler = choisir_echelle(X_train[SP_NUM])
echelle_train = transformer_Scaler.fit_transform(X_train[SP_NUM])
echelle_test = transformer_Scaler.transform(X_test[SP_NUM])
print("échelle choisie :", {n: sum(v == n for v in choix.values()) for n in ("standard", "robuste")})
# Fin de la solution
assert list(codeur_cible.classes_) == ["conforme", "non-conforme"]
assert np.bincount(yq_code).tolist() == [286, 434]
assert np.bincount(yq_test_code).tolist() == [71, 109]
assert EXCLURE_Q == ["id_soudure", "type_gaz_protection", "controle_visuel_resultat"]
assert len(textes) == 7
assert modalites == onehot_train.shape[1] == 31
assert onehot_test.shape[1] == 31
assert ord_train.shape == (720, 7)
assert onehot_train.shape[0] == 720
assert onehot_train.nnz == 720 * 7
assert Xq["type_assemblage"].isna().sum() == 4 and manuel_train["type_assemblage"].min() == 0
assert manuel_test["type_assemblage"].min() >= 0
assert manuel_train["site_fabrication"].isna().sum() == 0
assert abs(pentes["unités de départ"][1] - 0.9243) < 0.00005, \
    "r identique quelles que soient les unités"
assert all(abs(r - 0.9243) < 0.00005 for p, r in pentes.values())
assert abs(pentes["centré réduit"][0] - 0.9243) < 0.00005, "centrée réduite, la pente vaut r"
# Solution :
pente_forte = pentes["x divisé par 1000, y fois 1000"][0]
pente_faible = pentes["x fois 1000, y divisé par 1000"][0]
# Fin de la solution
assert pente_forte > 1e8 > 1e-3 > pente_faible
assert mm_train.min().min() == 0 and mm_train.max().max() <= 1 + 1e-12
assert round(mm_test["nombre_capteurs_installes"].max(), 3) == 0.680, \
    "le maximum du test reste dans [0 ; 1] ici"
assert np.abs(st_train.mean()).max() < 1e-9 and np.abs(st_test.mean()).max() > 0.001
assert sum(v == "robuste" for v in choix.values()) == 6 and echelle_train.shape == (16000, 9)
assert echelle_test.shape == (4000, 9)
print("Exercice 18.2 : contrôles passés")

cible encodée : ['conforme', 'non-conforme']  effectifs : [286, 434]  test : [71, 109]
colonnes de texte : 7 ; one-hot : 31 colonnes (matrice creuse, 5040 valeurs non nulles) ; ordinal : 7 colonnes
encodage manuel : mode de type_assemblage = recouvrement, de site_fabrication = Le Havre
  x fois 1000, y divisé par 1000     pente 3.809e-04   r 0.9243
  x divisé par 1000, y fois 1000     pente 3.809e+08   r 0.9243
  unités de départ                   pente 3.809e+02   r 0.9243
  centré réduit                      pente 9.243e-01   r 0.9243
min-max : étendue apprise de nombre_capteurs_installes [5 ; 30], maximum du test mis à l'échelle 0.680
standard : moyenne apprise de l'ancienneté 21.83 an, écart type 10.18 an
échelle choisie : {'standard': 3, 'robuste': 6}
Exercice 18.2 : contrôles passés


## Exercice 18.3 : La dispersion rétrécie par l'imputation (20 min)

Dans l'apprentissage du contrôle de soudures, imputez l'expérience du soudeur par la moyenne, la médiane et la constante $99{,}99$, puis par les voisins sur six colonnes numériques, avant et après une mise à l'échelle min-max. Pour chaque cas, divisez l'écart-type après imputation par l'écart-type avant.

**Résultat attendu.** 118 cases vides à l'apprentissage, 34 au test ; écart-type de 6,73 ans avant ; rapports de 0,914 (moyenne 8,22 ans), 0,921 (médiane), 5,14 (constante), 0,967 (voisins) et 0,976 (voisins mis à l'échelle).

**Seconde voie.** Avec la fraction $p$ de cases vides, $\sqrt{1-p}$ donne le rapport de la moyenne à quelques millièmes près.

**Indication.** Les six colonnes numériques sont l'expérience du soudeur, le nombre de passes, le préchauffage, le courant, l'épaisseur de tôle et la vitesse de soudage. Divisez l'écart-type de la colonne imputée par celui de la colonne d'origine. Pour les voisins mis à l'échelle, imputez les colonnes passées en min-max, puis revenez aux unités par `inverse_transform`.

In [9]:
colonnes_q = ["experience_soudeur_ans", "nb_passes_soudure", "temperature_prechauffage_C", "intensite_courant_A",
              "epaisseur_tole_mm", "vitesse_soudage_mm_min"]
# Solution :
Xqn, Xqn_test = Xq[colonnes_q], Xq_test[colonnes_q]
exp = "experience_soudeur_ans"
print("manquants de %s : %d en apprentissage, %d en test ; dispersion avant : %.3f an"
      % (exp, Xqn[exp].isna().sum(), Xqn_test[exp].isna().sum(), Xqn[exp].std()))
rapports = {}

for nom, options in [("mean", {}), ("median", {}), ("constant", {"fill_value": 99.99})]:
    imputeur, I_train, I_test = imputer_statistique(Xqn, Xqn_test, nom, **options)
    rapports[nom] = (imputeur.statistics_[0], I_train[exp].std() / Xqn[exp].std())
    print("  %-9s valeur apprise %.3f   dispersion après / avant : %.3f" % (nom, *rapports[nom]))

imp_k, K_train, K_test = imputer_voisins(Xqn, Xqn_test)
mm_q, M_train, M_test = mise_echelle_minmax(Xqn, Xqn_test)
imp_ks, KS_train, KS_test = imputer_voisins(M_train, M_test)
retour = pd.DataFrame(mm_q.inverse_transform(KS_train), index=KS_train.index, columns=KS_train.columns)
ratio_k, ratio_ks = K_train[exp].std() / Xqn[exp].std(), retour[exp].std() / Xqn[exp].std()
print("  voisins    dispersion après / avant : %.3f ; après mise à l'échelle des colonnes : %.3f" % (ratio_k, ratio_ks))
indicateur, T_train, T_test = indicateur_manquants(Xqn, Xqn_test)
print("indicateur de manquants :", T_train.columns.tolist(), " lignes à 1 : %d en apprentissage, %d en test"
      % (T_train.iloc[:, 0].sum(), T_test.iloc[:, 0].sum()))
imp_sp, S_train, S_test = imputer_statistique(X_train[SP_NUM], X_test[SP_NUM], "median")
mediane_test = X_test[SP_NUM].median().to_numpy()
print("suivi de production : manquants après imputation : %d en apprentissage, %d en test"
      % (S_train.isna().sum().sum(), S_test.isna().sum().sum()))
# Fin de la solution
assert Xqn[exp].isna().sum() == 118
assert Xqn_test[exp].isna().sum() == 34
assert Xqn.drop(columns=exp).isna().sum().sum() == 0
assert I_train.isna().sum().sum() == 0 and I_test.isna().sum().sum() == 0
assert abs(rapports["mean"][0] - 8.2174) < 0.00005 and abs(rapports["median"][0] - 6.2) < 1e-9
assert 0.91 < rapports["mean"][1] < 0.92 and 5.1 < rapports["constant"][1] < 5.2
assert ratio_k > rapports["mean"][1] and ratio_ks > ratio_k and ratio_ks < 1.0
assert (T_train.iloc[:, 0].sum(), T_test.iloc[:, 0].sum()) == (118, 34)
assert T_train.shape == (720, 1)
assert np.allclose(imp_sp.statistics_, X_train[SP_NUM].median().to_numpy()), \
    "médianes de l'apprentissage"
assert not np.allclose(imp_sp.statistics_, mediane_test), "les médianes du test ne servent à rien"
assert S_train.isna().sum().sum() == 0 and S_test.isna().sum().sum() == 0
print("Exercice 18.3 : contrôles passés")

manquants de experience_soudeur_ans : 118 en apprentissage, 34 en test ; dispersion avant : 6.725 an
  mean      valeur apprise 8.217   dispersion après / avant : 0.914
  median    valeur apprise 6.200   dispersion après / avant : 0.921
  constant  valeur apprise 99.990   dispersion après / avant : 5.137
  voisins    dispersion après / avant : 0.967 ; après mise à l'échelle des colonnes : 0.976
indicateur de manquants : ['manquant_experience_soudeur_ans']  lignes à 1 : 118 en apprentissage, 34 en test
suivi de production : manquants après imputation : 0 en apprentissage, 0 en test
Exercice 18.3 : contrôles passés


## Exercice 18.4 : Un filtre de variance sur le suivi de production (20 min)

Sur les colonnes numériques du suivi de production, avec l'énergie par pièce, imputez par la médiane, mettez à l'échelle en min-max, puis filtrez au seuil $0{,}005$. Quelles colonnes sont écartées, quelle est la plus faible variance ?

**Résultat attendu.** 4 colonnes écartées : la consommation électrique, le niveau de complexité, le nombre de capteurs et le taux d'utilisation ; variance minimale de 0,00009.

**Seconde voie.** Les variances du tableau mis à l'échelle, calculées avec `.var()`, passent sous le seuil pour ces quatre colonnes.

**Indication.** Imputez avec `imputer_statistique` et la médiane, mettez à l'échelle avec `mise_echelle_minmax`, puis appelez `filtrer_variance` avec `seuil=0.005`. Une valeur extrême étire l'étendue : le reste tient dans un coin de $[0\,;1]$ et la variance chute.

In [10]:
# Solution :
X5_train = preprocessing_grandeurs(df_train, EXCLURE)
X5_test = preprocessing_grandeurs(df_test, EXCLURE)
Xe_train, ye_train = preprocessing_Xy(X5_train, CIBLE)
Xe_test, ye_test = preprocessing_Xy(X5_test, CIBLE)
energie = Xe_train["energie_par_piece_kWh"]
print("énergie par pièce : médiane %.3f kWh, %d manquants (ceux de la consommation)" % (energie.median(), energie.isna().sum()))
duo = ["anciennete_machine_en_annees", "indice_efficacite_energetique"]
poly, P_train, P_test = termes_polynomiaux(Xe_train[duo], Xe_test[duo])
prod = ["productivite_moyenne_en_pieces_par_heure"]
power, Y_train, Y_test = transformer_distribution(Xe_train[prod], Xe_test[prod])
classes, C_train, C_test = decouper_en_classes(Xe_train[duo[:1]], Xe_test[duo[:1]])
binaire, B_train, B_test = decouper_en_classes(Xe_train[duo[:1]], Xe_test[duo[:1]], seuil=30.0)

def custom_func(x):
    """Fonction choisie : décroissante, bornée, sans valeur négative sous la racine."""
    return 1 / (15 + np.sqrt(np.clip(x, 0, None)))

def custom_func_inv(y):
    return (1 / y - 15) ** 2

vib = ["niveau_vibrations_en_mm_par_s"]
fonction, F_train, F_test = transformer_fonction(Xe_train[vib], Xe_test[vib], custom_func, custom_func_inv)
retour_vib = fonction.inverse_transform(F_train)
ecart_retour = np.nanmax(np.abs(retour_vib.to_numpy() - Xe_train[vib].to_numpy()))
print("termes polynomiaux :", P_train.shape[1], "colonnes ;", "asymétrie de la productivité : %.3f avant, %.3f après"
      % (Xe_train[prod[0]].skew(), Y_train.iloc[:, 0].skew()))
print("quatre classes d'ancienneté, effectifs :", C_train.iloc[:, 0].value_counts().sort_index().astype(int).tolist(),
      " bornes (années) :", classes.bin_edges_[0].round(1).tolist())
print("plus de 30 ans : %.1f %% en apprentissage, %.1f %% en test" % (100 * B_train.iloc[:, 0].mean(), 100 * B_test.iloc[:, 0].mean()))
print("fonction de l'ingénieur : retour aux unités de départ à %.1e mm/s près" % ecart_retour)
# Filtre de variance sur les colonnes mises à l'échelle min-max : imputer, normaliser, puis filtrer.
SP_NUM5 = Xe_train.select_dtypes("number").columns.tolist()
_, J_train, J_test = imputer_statistique(Xe_train[SP_NUM5], Xe_test[SP_NUM5], "median")
_, N_train, N_test = mise_echelle_minmax(J_train, J_test)
variances = N_train.var().sort_values()
filtre, V_train, V_test = filtrer_variance(N_train, N_test, seuil=0.005)
ecartees = sorted(set(N_train.columns) - set(V_train.columns))
print("variance après min-max, trois plus faibles :", variances.round(5).head(3).to_dict())
print("colonnes écartées par le filtre de variance :", ecartees)
# Sélection sur la surveillance vibratoire : cible numérique, seize entrées, 120 lignes d'apprentissage.
data_sv, sv = lire_classeur("MecaIA-Surveillance_Vibratoire.xlsx")
cible_sv = "duree_vie_restante_RUL_h"
sv_train, sv_test = decouper(sv)
entrees = [c for c in sv.select_dtypes("number").columns if c not in (cible_sv, "alarme_criticite")]
_, SV_a, SV_t = imputer_statistique(sv_train[entrees], sv_test[entrees], "median")
_, SV_n, SV_nt = mise_echelle_minmax(SV_a, SV_t)
y_sv = sv_train[cible_sv]
sel_score, K4_train, K4_test = selection_score(SV_n, y_sv, SV_nt, k=4)
sel_modele, M4_train, M4_test = selection_modele(SV_n, y_sv, SV_nt)
rfe, R_train, R_test = elimination_recursive(SV_n, y_sv, SV_nt, n=4)
rfecv, RC_train, RC_test = elimination_recursive(SV_n, y_sv, SV_nt, n=4, cv=6)
classement = pd.Series(sel_score.scores_, index=SV_n.columns).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(3.4, 2.4))
scores_k = controle_selection(SV_n, y_sv, classement.index.tolist(), ax=ax)
print("entrées : %d ; lignes d'apprentissage : %d ; manquants imputés : %d" % (len(entrees), len(sv_train), sv_train[entrees].isna().sum().sum()))
print("test univarié, quatre meilleures :", K4_train.columns.tolist())
print("sélection par le modèle :", M4_train.columns.tolist())
print("élimination récursive :", R_train.columns.tolist(), " RFECV garde", rfecv.n_features_, "colonnes")
print("R2 de validation selon le nombre de variables (rang du test univarié) :", np.round(scores_k[:4], 3).tolist(),
      "... meilleur à", int(np.argmax(scores_k)) + 1, "variables : %.3f" % max(scores_k))
# Fin de la solution
assert energie.isna().sum() == 1605 and abs(energie.median() - 0.182) < 0.0005
# Solution :
noms_poly = ["anciennete_machine_en_annees", "indice_efficacite_energetique", "anciennete_machine_en_annees^2",
             "anciennete_machine_en_annees indice_efficacite_energetique", "indice_efficacite_energetique^2"]
# Fin de la solution
assert P_train.columns.tolist() == noms_poly
assert abs(Xe_train[prod[0]].skew() - 1.673) < 0.0005 and abs(Y_train.iloc[:, 0].skew()) < 0.05
assert C_train.iloc[:, 0].value_counts().sort_index().tolist() == [2696, 4776, 5184, 3344]
assert round(B_train.iloc[:, 0].mean(), 4) == 0.2284
assert round(B_test.iloc[:, 0].mean(), 4) == 0.2428
assert ecart_retour < 1e-9 and round(F_train.iloc[:, 0].std(), 4) == 0.0011
assert ecartees == ["consommation_electrique_en_kWh_par_heure", "niveau_complexite_machine",
                    "nombre_capteurs_installes", "taux_utilisation_machine"]
assert V_train.shape == (16000, 6) and V_test.shape == (4000, 6) and variances.iloc[0] < 0.0001
assert len(entrees) == 16 and K4_train.columns.tolist() == [
    "age_machine_annees", "vibration_bande_1X_mm_s", "vibration_bande_HF_mm_s", "indice_degradation"]
assert classement.index[0] == "indice_degradation" and classement.index[1] == "age_machine_annees"
assert M4_train.columns.tolist() == R_train.columns.tolist(), \
    "modèle et élimination récursive s'accordent"
assert "taux_charge_ratio" in M4_train.columns
assert "charge_radiale_kN" in M4_train.columns, "variables collinéaires"
assert rfecv.n_features_ == 4 and len(scores_k) == 16
assert scores_k[0] > 0.94
assert scores_k[1] > 0.98
assert max(scores_k) - scores_k[1] < 0.01, "plateau dès deux variables"
# Solution :
plt.close(fig)
# Fin de la solution
print("Exercice 18.4 : contrôles passés")

énergie par pièce : médiane 0.182 kWh, 1605 manquants (ceux de la consommation)
termes polynomiaux : 5 colonnes ; asymétrie de la productivité : 1.673 avant, 0.007 après
quatre classes d'ancienneté, effectifs : [2696, 4776, 5184, 3344]  bornes (années) : [1.0, 11.4, 21.1, 30.7, 40.0]
plus de 30 ans : 22.8 % en apprentissage, 24.3 % en test
fonction de l'ingénieur : retour aux unités de départ à 8.9e-15 mm/s près
variance après min-max, trois plus faibles : {'taux_utilisation_machine': 9e-05, 'nombre_capteurs_installes': 0.00263, 'niveau_complexite_machine': 0.00304}
colonnes écartées par le filtre de variance : ['consommation_electrique_en_kWh_par_heure', 'niveau_complexite_machine', 'nombre_capteurs_installes', 'taux_utilisation_machine']
entrées : 16 ; lignes d'apprentissage : 120 ; manquants imputés : 88
test univarié, quatre meilleures : ['age_machine_annees', 'vibration_bande_1X_mm_s', 'vibration_bande_HF_mm_s', 'indice_degradation']
sélection par le modèle : ['vitesse_rotation_rp

## Exercice 18.5 : Un score avec et sans fuite (défi, 30 min)

Sur le suivi de production, avec l'énergie par pièce, mesurez avec cinq voisins et les colonnes complètes l'exactitude d'une référence, d'une standardisation avant la validation croisée, puis d'un `Pipeline`. Réglez le nombre de voisins d'un `preparer` par `GridSearchCV`, comptez la classe majoritaire, ouvrez le test une fois. Montrez enfin deux fuites : l'étendue de l'énergie par pièce sur tout le jeu, et une sélection sur du bruit.

**Résultat attendu.** 58,4 %, 58,9 % et 58,8 % ; chaîne par branches avec 61 voisins, validation 79,8 %, test 79,0 %, contre 56,8 % pour la classe majoritaire ; énergie par pièce : maximum de 0,665 kWh sur tout le jeu contre 0,660 à l'apprentissage (0,8 % de l'étendue) ; sélection sur du bruit : $R^2$ de 0,084 avec fuite, $-0{,}61$ sans.

**Seconde voie.** Une cible mélangée au hasard ne contient aucun signal : un score positif isolé peut venir du hasard, et l'on répète les permutations pour juger si le score observé est inhabituel.

**Indication.** La référence, la standardisation et le `Pipeline` sont trois appels de `cross_val_score` à cinq plis ; pour la chaîne par branches, codez la cible avec `encoder_cible`, puis assemblez dans un `Pipeline` la préparation de `preparer`, appelée avec les noms des colonnes numériques et de texte, et un classifieur à plus proches voisins, avec une grille de nombres de voisins ; notez le test une seule fois. Pour la fuite de la sélection, ajoutez deux cents colonnes de bruit, mélangez la cible, sélectionnez dix colonnes avec toutes les lignes (fuite), puis dans un `make_pipeline` (sans fuite), et validez : le score est positif avec la fuite, nettement négatif sans.

In [11]:
# Solution :
for temps, geste in workflow_quatre_temps():
    print("  %-34s %s" % (temps, geste))

connu = KNeighborsClassifier(n_neighbors=5)
Xc = Xe_train[complets]
score_ref = cross_val_score(connu, Xc, ye_train, cv=5).mean()
Zc = StandardScaler().fit_transform(Xc)                          # échelle apprise avant la validation croisée
score_sans_pipeline = cross_val_score(connu, Zc, ye_train, cv=5).mean()
score_pipeline = cross_val_score(pipeline_simple(connu, Xc, ye_train), Xc, ye_train, cv=5).mean()
print("1 référence : %.4f ; 2 standardisation sans pipeline : %.4f ; 3 pipeline : %.4f"
      % (score_ref, score_sans_pipeline, score_pipeline))
num_cols = Xe_train.select_dtypes("number").columns.tolist()
cat_cols = Xe_train.columns.drop(num_cols).tolist()
preparation = preparer(num_cols, cat_cols, score=f_classif)

for ligne in decrire_pipeline(preparation):
    print("  " + ligne)

grille = {"classifier__n_neighbors": [5, 15, 31, 61, 121, 241]}
codeur_y, ye_code, ye_test_code = encoder_cible(ye_train, ye_test)     # la sélection de `preparer` attend des nombres
recherche = GridSearchCV(Pipeline([("preparer", preparation), ("classifier", KNeighborsClassifier())]), grille, cv=5)
recherche.fit(Xe_train, ye_code)
meilleur_k = recherche.best_params_["classifier__n_neighbors"]
print("4 amélioration : meilleur nombre de voisins %d, exactitude de validation %.4f" % (meilleur_k, recherche.best_score_))
print("part de la classe majoritaire à battre (apprentissage) : %.4f" % ye_train.value_counts(normalize=True).max())
# La fuite, vue sur les étendues : le test aurait fixé le maximum de l'énergie par pièce.
tout = pd.concat([Xe_train, Xe_test])
numeriques5 = Xe_train.select_dtypes("number").columns
ecarts = {c: (tout[c].max() - Xe_train[c].max()) / (tout[c].max() - tout[c].min()) for c in numeriques5}
col_fuite = max(ecarts, key=ecarts.get)
print("fuite par l'échelle : %s, maximum %.3f sur tout le jeu contre %.3f sur l'apprentissage (%.1f %% de l'étendue)"
      % (col_fuite, tout[col_fuite].max(), Xe_train[col_fuite].max(), 100 * ecarts[col_fuite]))
# La fuite par la sélection, visible sur une cible mélangée et 200 colonnes de bruit : aucun modèle honnête ne dépasse 0.
generateur = np.random.default_rng(0)
bruit = pd.DataFrame(generateur.normal(size=(len(sv_train), 200)), index=sv_train.index,
                     columns=["bruit_%03d" % i for i in range(200)])
y_melange = pd.Series(generateur.permutation(y_sv.to_numpy()), index=y_sv.index)
r2_fuite = cross_val_score(LinearRegression(), SelectKBest(f_regression, k=10).fit_transform(bruit, y_melange),
                           y_melange, cv=5).mean()
r2_propre = cross_val_score(make_pipeline(SelectKBest(f_regression, k=10), LinearRegression()), bruit, y_melange,
                            cv=5).mean()
print("sélection sur du bruit : R2 %.3f avec fuite, %.3f sans fuite" % (r2_fuite, r2_propre))
exactitude_test = recherche.score(Xe_test, ye_test_code)                # le test, ouvert une seule fois
print("test ouvert une fois : exactitude %.4f" % exactitude_test)
print("ordre d'emploi des opérations :", " puis ".join(ORDRE_OPERATIONS))
print("ordre des briques de la préparation (celui de Méca-IA) :", " puis ".join(
    i for i, n, s, a in ORDRE_EMPLOI if i.split("-")[0] in ("DEC", "X", "I", "E", "N", "S", "A") and i != "DEC-01"))
# Fin de la solution
assert [t for t, g in workflow_quatre_temps()][0] == "1 Référence"
assert len(workflow_quatre_temps()) == 4
assert abs(score_ref - 0.5839) < 0.00005 and abs(score_pipeline - 0.5882) < 0.00005
assert abs(score_sans_pipeline - score_pipeline) < 0.005, "ici l'échelle seule fuit très peu"
# Solution :
lignes_pipeline = decrire_pipeline(preparation)
# Fin de la solution
assert lignes_pipeline[0] == "num_cols_preproc : SimpleImputer puis StandardScaler sur 10 colonnes"
assert lignes_pipeline[1] == "cat_cols_preproc : SimpleImputer puis OneHotEncoder sur 1 colonnes"
assert meilleur_k >= 61 and recherche.best_score_ > score_pipeline + 0.04
assert col_fuite == "energie_par_piece_kWh"
assert abs(tout[col_fuite].max() - 0.6651785714285715) < 1e-12
assert abs(Xe_train[col_fuite].max() - 0.660287081339713) < 1e-12
assert r2_fuite > 0.05 and r2_propre < -0.3, "la fuite fait passer du bruit pour un signal"
assert exactitude_test > ye_test.value_counts(normalize=True).max() + 0.1
print("Exercice 18.5 : contrôles passés")

  1 Référence                        colonnes complètes, découpage, modèle simple : le score à battre
  2 Standardisation sans pipeline    échelle apprise sur tout l'apprentissage avant la validation croisée : les plis la partagent
  3 Pipeline                         l'échelle est apprise dans chaque pli : score sans fuite
  4 Boucle d'amélioration            branches numérique et catégorielle, grille d'hyperparamètres, retour à l'EDA
1 référence : 0.5839 ; 2 standardisation sans pipeline : 0.5886 ; 3 pipeline : 0.5882
  num_cols_preproc : SimpleImputer puis StandardScaler sur 10 colonnes
  cat_cols_preproc : SimpleImputer puis OneHotEncoder sur 1 colonnes
  sélection des k meilleures colonnes (SelectKBest)
4 amélioration : meilleur nombre de voisins 61, exactitude de validation 0.7976
part de la classe majoritaire à battre (apprentissage) : 0.5679
fuite par l'échelle : energie_par_piece_kWh, maximum 0.665 sur tout le jeu contre 0.660 sur l'apprentissage (0.8 % de l'étendue)
sélection

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F18-1 : Mise à l'échelle (`sklearn.preprocessing.StandardScaler`)

In [12]:
import numpy as np
from sklearn.preprocessing import StandardScaler

X = np.array([[157.0, 20.0], [167.0, 25.0], [151.0, 18.0], [178.0, 28.0]])
scaler = StandardScaler()
X_mis = scaler.fit_transform(X)           # centrage et réduction
attendu = (X - np.mean(X, axis=0)) / np.std(X, axis=0)
assert np.allclose(X_mis, attendu)
print(" ".join(f"{x:.3f}" for x in X_mis[0]).replace(".", ","))
print(" ".join(f"{x:.1f}" for x in scaler.mean_).replace(".", ","))

-0,609 -0,694
163,2 22,8


### Fiche F18-2 : Chaîne de préparation (`sklearn.pipeline.Pipeline`)

In [13]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

rng = np.random.default_rng(0)
X = np.column_stack([rng.uniform(100, 200, 20), rng.uniform(0.1, 0.3, 20)])
y = 50 * X[:, 1]**2 + 0.005 * X[:, 0] + rng.normal(0, 0.1, 20)
pipe = Pipeline([("e", StandardScaler()), ("m", Ridge())]).fit(X, y)
print(f"Ra prévue : {pipe.predict([[160, 0.15]])[0]:.2f} µm".replace(".", ","))
print(np.isclose(pipe.named_steps["e"].mean_[0], np.mean(X[:, 0])))

Ra prévue : 1,98 µm
True
